In [1]:
pip install pandas numpy scikit-learn tensorflow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
pip install gensim

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Import library

In [3]:
import os
import re
import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

# Project paths: force all files to stay inside Senior_Project_Text_Mining
BASE_DIR = Path.cwd()

if BASE_DIR.name != "Senior_Project_Text_Mining":
    possible_dir = BASE_DIR / "Senior_Project_Text_Mining"
    if possible_dir.exists():
        BASE_DIR = possible_dir
    else:
        raise FileNotFoundError(
            "Please open or run this notebook from the Senior_Project_Text_Mining folder."
        )

DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR.resolve())
print("RAW_DIR:", RAW_DIR.resolve())
print("PROCESSED_DIR:", PROCESSED_DIR.resolve())


BASE_DIR: C:\3rd Year\Senior Project\Emotion_Analysis\Senior_Project_Text_Mining
RAW_DIR: C:\3rd Year\Senior Project\Emotion_Analysis\Senior_Project_Text_Mining\data\raw
PROCESSED_DIR: C:\3rd Year\Senior Project\Emotion_Analysis\Senior_Project_Text_Mining\data\processed


### Load Dataset

In [4]:
file_path = RAW_DIR / "All_emotions.csv"
df = pd.read_csv(file_path)

print("Loaded from:", file_path.resolve())
print("Dataset shape:", df.shape)
df.head()


Loaded from: C:\3rd Year\Senior Project\Emotion_Analysis\Senior_Project_Text_Mining\data\raw\All_emotions.csv
Dataset shape: (7261, 3)


,speaker,utterance,emotion
0,patient,I can feel the cold water on my teeth.,neutral
1,dentist,It's important to keep the area clean.,neutral
2,patient,I try to brush twice a day like you said.,neutral
3,dentist,You might feel a bit of pressure.,neutral
4,patient,Should I open wider?,neutral


### Select Patient Utterances

In [5]:
df.columns = df.columns.str.strip().str.lower()

df = df[df["speaker"].str.lower() == "patient"].copy()

print("Filtered dataset shape:", df.shape)
df.head()

Filtered dataset shape: (3636, 3)


,speaker,utterance,emotion
0,patient,I can feel the cold water on my teeth.,neutral
2,patient,I try to brush twice a day like you said.,neutral
4,patient,Should I open wider?,neutral
6,patient,I am not paying for a mistake you made.,angry
8,patient,I can feel the cold water on my teeth.,neutral


### Clean Text

In [6]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["utterance"].apply(clean_text)

df = df[df["clean_text"] != ""].copy()

df.head()

,speaker,utterance,emotion,clean_text
0,patient,I can feel the cold water on my teeth.,neutral,i can feel the cold water on my teeth
2,patient,I try to brush twice a day like you said.,neutral,i try to brush twice a day like you said
4,patient,Should I open wider?,neutral,should i open wider
6,patient,I am not paying for a mistake you made.,angry,i am not paying for a mistake you made
8,patient,I can feel the cold water on my teeth.,neutral,i can feel the cold water on my teeth


### Prepare Input Data

In [7]:
df = df[["clean_text", "emotion"]].dropna().copy()
df.columns = ["text", "label"]

print("Prepared dataset shape:", df.shape)
df.head()

Prepared dataset shape: (3636, 2)


,text,label
0,i can feel the cold water on my teeth,neutral
2,i try to brush twice a day like you said,neutral
4,should i open wider,neutral
6,i am not paying for a mistake you made,angry
8,i can feel the cold water on my teeth,neutral


### Check Label Distribution

In [8]:
print(df["label"].value_counts())

label
neutral    2856
disgust     247
fear        195
angry       156
sad          98
happy        84
Name: count, dtype: int64


### Save Cleaned Data

In [9]:
df.to_csv(PROCESSED_DIR / "cleaned_emotions_dl.csv", index=False)

print("Cleaned DL dataset saved successfully")
print("Saved to:", (PROCESSED_DIR / "cleaned_emotions_dl.csv").resolve())


Cleaned DL dataset saved successfully
Saved to: C:\3rd Year\Senior Project\Emotion_Analysis\Senior_Project_Text_Mining\data\processed\cleaned_emotions_dl.csv


### Split Dataset

In [10]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

Train shape: (2545, 2)
Validation shape: (545, 2)
Test shape: (546, 2)


### Tokenization

In [11]:
max_vocab_size = 10000

tokenizer = Tokenizer(num_words=max_vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df["text"])

word_index = tokenizer.word_index

print("Vocabulary size:", len(word_index))
print("Sample word index:", list(word_index.items())[:10])

Vocabulary size: 1780
Sample word index: [('<OOV>', 1), ('i', 2), ('the', 3), ('it', 4), ('my', 5), ('to', 6), ('that', 7), ('is', 8), ('a', 9), ('and', 10)]


### Convert Text to Sequences

In [12]:
X_train_seq = tokenizer.texts_to_sequences(train_df["text"])
X_val_seq = tokenizer.texts_to_sequences(val_df["text"])
X_test_seq = tokenizer.texts_to_sequences(test_df["text"])

print("Sample text:", train_df["text"].iloc[0])
print("Sample sequence:", X_train_seq[0])

Sample text: i am so happy everything looks better now
Sample sequence: [2, 23, 62, 70, 158, 168, 129, 48]


### Pad Sequences

In [13]:
sequence_lengths = [len(seq) for seq in X_train_seq]

print("Maximum sequence length:", max(sequence_lengths))
print("Average sequence length:", np.mean(sequence_lengths))
print("95th percentile length:", np.percentile(sequence_lengths, 95))

max_sequence_length = int(np.percentile(sequence_lengths, 95))
print("Chosen max sequence length:", max_sequence_length)

Maximum sequence length: 27
Average sequence length: 8.597642436149313
95th percentile length: 16.0
Chosen max sequence length: 16


In [14]:
X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=max_sequence_length,
    padding="post",
    truncating="post"
)

X_val_pad = pad_sequences(
    X_val_seq,
    maxlen=max_sequence_length,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=max_sequence_length,
    padding="post",
    truncating="post"
)

print("X_train_pad shape:", X_train_pad.shape)
print("X_val_pad shape:", X_val_pad.shape)
print("X_test_pad shape:", X_test_pad.shape)

X_train_pad shape: (2545, 16)
X_val_pad shape: (545, 16)
X_test_pad shape: (546, 16)


### Encode Labels

In [15]:
label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(train_df["label"])
y_val = label_encoder.transform(val_df["label"])
y_test = label_encoder.transform(test_df["label"])

print("Classes:", label_encoder.classes_)
print("Sample encoded labels:", y_train[:10])

Classes: ['angry' 'disgust' 'fear' 'happy' 'neutral' 'sad']
Sample encoded labels: [3 4 4 1 0 4 4 4 4 4]


### One-Hot Encode Labels

In [16]:
num_classes = len(label_encoder.classes_)

y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_val_cat = to_categorical(y_val, num_classes=num_classes)
y_test_cat = to_categorical(y_test, num_classes=num_classes)

print("y_train_cat shape:", y_train_cat.shape)
print("y_val_cat shape:", y_val_cat.shape)
print("y_test_cat shape:", y_test_cat.shape)

y_train_cat shape: (2545, 6)
y_val_cat shape: (545, 6)
y_test_cat shape: (546, 6)


### Save Processed Data

In [17]:
np.save(PROCESSED_DIR / "X_train_pad.npy", X_train_pad)
np.save(PROCESSED_DIR / "X_val_pad.npy", X_val_pad)
np.save(PROCESSED_DIR / "X_test_pad.npy", X_test_pad)

np.save(PROCESSED_DIR / "y_train.npy", y_train)
np.save(PROCESSED_DIR / "y_val.npy", y_val)
np.save(PROCESSED_DIR / "y_test.npy", y_test)

np.save(PROCESSED_DIR / "y_train_cat.npy", y_train_cat)
np.save(PROCESSED_DIR / "y_val_cat.npy", y_val_cat)
np.save(PROCESSED_DIR / "y_test_cat.npy", y_test_cat)

print("Processed arrays saved successfully")


Processed arrays saved successfully


### Save Tokenizer and Label Mapping

In [18]:
with open(PROCESSED_DIR / "word_index.json", "w", encoding="utf-8") as f:
    json.dump(tokenizer.word_index, f, ensure_ascii=False, indent=2)

label_mapping = {
    label: int(index)
    for index, label in enumerate(label_encoder.classes_)
}

with open(PROCESSED_DIR / "label_mapping.json", "w", encoding="utf-8") as f:
    json.dump(label_mapping, f, ensure_ascii=False, indent=2)

print("Tokenizer word index and label mapping saved successfully")
print("Label mapping:", label_mapping)


Tokenizer word index and label mapping saved successfully
Label mapping: {'angry': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 'neutral': 4, 'sad': 5}


### Final Summary

In [19]:
print("Deep learning preprocessing completed successfully")
print("Final training input shape:", X_train_pad.shape)
print("Final validation input shape:", X_val_pad.shape)
print("Final test input shape:", X_test_pad.shape)
print("Number of classes:", num_classes)
print("Vocabulary size:", len(tokenizer.word_index))
print("Max sequence length:", max_sequence_length)

print("\nSaved files in processed folder:")
for file in sorted(PROCESSED_DIR.iterdir()):
    print("-", file.name)


Deep learning preprocessing completed successfully
Final training input shape: (2545, 16)
Final validation input shape: (545, 16)
Final test input shape: (546, 16)
Number of classes: 6
Vocabulary size: 1780
Max sequence length: 16

Saved files in processed folder:
- cleaned_emotions_dl.csv
- cleaned_emotions_patient.csv
- label_mapping.json
- patient_test_tfidf.csv
- patient_train_tfidf_smote.csv
- tfidf_features_patient.csv
- word_index.json
- X_test_pad.npy
- X_train_pad.npy
- X_val_pad.npy
- y_test.npy
- y_test_cat.npy
- y_train.npy
- y_train_cat.npy
- y_val.npy
- y_val_cat.npy
